<a href="https://colab.research.google.com/github/KishanGollamudi/Forward-Deployment-Project/blob/feature%2Ftpu-automation-pipeline/tpu_llm_serving_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch_xla.core.xla_model as xm

# Query connected TPU hardware core
device = xm.xla_device()
devices = xm.get_xla_supported_devices()

print("=" * 50)
print("TPU Hardware Status: CONNECTED")
print(f"Target Execution Device: {device}")
print(f"Discovered XLA Cores   : {devices}")
print("=" * 50)

ModuleNotFoundError: No module named 'torch_xla'

In [ ]:
import torch

# Query the native TPU device registered by the Colab runtime
device = torch.device("tpu")
print("=" * 45)
print("TPU Hardware Status: CONNECTED & READY")
print(f"Target Execution Device: {device}")
print("=" * 45)

# Run a quick matrix multiplication directly on the TPU tensor core
x = torch.randn((4, 4), device=device)
y = torch.ones((4, 4), device=device)
z = torch.matmul(x, y)

print("\nTPU Tensor Computation Result:")
print(z)

In [ ]:
!pip install -q transformers accelerate sentencepiece

In [ ]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device("tpu")
print(f"Allocating execution context on Cloud TPU v5e ({device})...")

# 1. Download tokenizer & weights directly to TPU memory in bfloat16
model_id = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16
).to(device)

# 2. Set up the prompt
prompt = "Explain in 3 concise bullet points why systolic arrays excel at dense GEMM operations on TPUs."
messages = [
    {"role": "system", "content": "You are an expert AI infrastructure engineer."},
    {"role": "user", "content": prompt}
]
formatted_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted_input, return_tensors="pt").to(device)

# 3. Execution & timing
print("\nCompiling graph and generating tokens on TPU v5e...")
start_time = time.time()

with torch.no_grad():
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False
    )

elapsed = time.time() - start_time
response = tokenizer.decode(output_tokens[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print(f"\n✅ Execution completed in {elapsed:.2f}s!")
print("-" * 50)
print(response)

In [ ]:
# 1. Warm inference test
bench_prompt = "Compare Kubernetes DaemonSets vs Deployments for edge observability in 2 concise bullets."
messages = [
    {"role": "system", "content": "You are a Cloud Infrastructure Engineer."},
    {"role": "user", "content": bench_prompt}
]
formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted, return_tensors="pt").to(device)

start_time = time.time()
with torch.no_grad():
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

elapsed = time.time() - start_time
tokens_generated = len(output_tokens[0]) - inputs.input_ids.shape[1]
tokens_per_sec = tokens_generated / elapsed
response = tokenizer.decode(output_tokens[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print(f"⚡ Warm TPU Generation: {elapsed:.2f}s | Speed: {tokens_per_sec:.1f} tokens/sec")
print("-" * 50)
print(response)

In [ ]:
# 1. Define multiple simultaneous inference requests
batch_prompts = [
    "What is the role of an Ingress Controller in Kubernetes?",
    "Explain Blue-Green deployment strategy in one sentence.",
    "Why use bfloat16 over float32 on Cloud TPUs?",
    "What is the difference between horizontal and vertical pod autoscaling?"
]

# Set padding token
tokenizer.pad_token = tokenizer.eos_token
formatted_batch = [
    tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
    for p in batch_prompts
]
inputs = tokenizer(formatted_batch, return_tensors="pt", padding=True).to(device)

print(f"Executing batch size {len(batch_prompts)} concurrently on TPU v5e...")
start_time = time.time()

with torch.no_grad():
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False
    )

elapsed = time.time() - start_time
total_tokens = sum(len(out) - len(inp) for out, inp in zip(output_tokens, inputs.input_ids))
effective_tps = total_tokens / elapsed

print(f"⚡ Batched Execution: {elapsed:.2f}s | Aggregate Throughput: {effective_tps:.1f} tokens/sec")
print("=" * 60)

for idx, out in enumerate(output_tokens):
    resp = tokenizer.decode(out[inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\n[Prompt {idx+1}] {batch_prompts[idx]}")
    print(f"[Response] {resp.strip()}")

In [ ]:
# 1. Fix decoder padding orientation
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

batch_prompts = [
    "What is the role of an Ingress Controller in Kubernetes?",
    "Explain Blue-Green deployment strategy in one sentence.",
    "Why use bfloat16 over float32 on Cloud TPUs?",
    "What is the difference between horizontal and vertical pod autoscaling?"
]

formatted_batch = [
    tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
    for p in batch_prompts
]

# Pad cleanly to left with fixed tensor alignment
inputs = tokenizer(
    formatted_batch,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=128
).to(device)

print("Running left-padded batch generation on TPU v5e...")
start_time = time.time()

with torch.no_grad():
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

elapsed = time.time() - start_time
total_tokens = sum(len(out) - len(inp) for out, inp in zip(output_tokens, inputs.input_ids))
effective_tps = total_tokens / elapsed

print(f"⚡ Clean Batch Execution: {elapsed:.2f}s | Speed: {effective_tps:.1f} tokens/sec")
print("=" * 60)

for idx, out in enumerate(output_tokens):
    # Slice off prompt input to get pure generation
    resp = tokenizer.decode(out[inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\n[Prompt {idx+1}] {batch_prompts[idx]}")
    print(f"[Response] {resp.strip()}")

In [ ]:
!pip install -q fastapi uvicorn pydantic pyngrok

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import threading
import uvicorn
import time

app = FastAPI(title="Cloud TPU v5e LLM Service", version="1.0.0")

class InferenceRequest(BaseModel):
    prompt: str
    max_tokens: int = 80
    temperature: float = 0.0

class InferenceResponse(BaseModel):
    response: str
    latency_sec: float
    device: str
    tokens_generated: int

@app.get("/healthz")
def healthcheck():
    return {"status": "healthy", "accelerator": "TPU v5e-1", "backend": str(device)}

@app.post("/v1/generate", response_model=InferenceResponse)
def generate_text(req: InferenceRequest):
    try:
        t0 = time.time()

        # Prepare structured input
        messages = [{"role": "user", "content": req.prompt}]
        formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(formatted, return_tensors="pt").to(device)

        with torch.no_grad():
            output_tokens = model.generate(
                **inputs,
                max_new_tokens=req.max_tokens,
                do_sample=(req.temperature > 0.0),
                pad_token_id=tokenizer.eos_token_id
            )

        elapsed = time.time() - t0
        new_tokens = output_tokens[0][inputs.input_ids.shape[1]:]
        decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        return InferenceResponse(
            response=decoded,
            latency_sec=round(elapsed, 3),
            device="tpu:0",
            tokens_generated=len(new_tokens)
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# Run server on background thread inside Colab
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("🚀 FastAPI LLM Server running locally on port 8000!")

In [ ]:
import requests
import json

time.sleep(2)  # Wait for uvicorn to bind

# 1. Probe healthcheck
health = requests.get("http://127.0.0.1:8000/healthz").json()
print("Health Probe:", health)

# 2. Issue an inference payload
payload = {
    "prompt": "What are three key metrics to monitor for an LLM inference service?",
    "max_tokens": 100
}
res = requests.post("http://127.0.0.1:8000/v1/generate", json=payload).json()
print("\nAPI Response:")
print(json.dumps(res, indent=2))

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr
import time
import torch

# Ensure left-padding is active for clean generation
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

def chat_tpu(message, history, max_new_tokens, temperature):
    # 1. Format the conversation using Qwen's chat template
    conversation = []
    for user_turn, bot_turn in history:
        conversation.append({"role": "user", "content": user_turn})
        if bot_turn:
            conversation.append({"role": "assistant", "content": bot_turn})
    conversation.append({"role": "user", "content": message})

    formatted_prompt = tokenizer.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=True
    )

    # 2. Tokenize and ship directly to the TPU core
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

    # 3. Hardware inference
    start_time = time.time()
    with torch.no_grad():
        output_tokens = model.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            do_sample=(temperature > 0.0),
            temperature=float(temperature) if temperature > 0.0 else None,
            pad_token_id=tokenizer.eos_token_id
        )
    elapsed = time.time() - start_time

    # 4. Decode new tokens only
    new_tokens = output_tokens[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    speed = len(new_tokens) / elapsed if elapsed > 0 else 0
    print(f"TPU Inference: {len(new_tokens)} tokens generated in {elapsed:.2f}s ({speed:.1f} tok/s)")

    return response

# Build the Web UI
demo = gr.ChatInterface(
    fn=chat_tpu,
    title="Cloud TPU v5e-1 Inference Dashboard",
    description="Interactive chatbot running on Google Cloud TPU v5e hardware via native PyTorch-XLA.",
    additional_inputs=[
        gr.Slider(minimum=20, maximum=256, value=100, step=10, label="Max New Tokens"),
        gr.Slider(minimum=0.0, maximum=1.0, value=0.0, step=0.1, label="Temperature (0 = Greedy)")
    ],
    theme="soft"
)

# share=True creates a public secure HTTPS URL accessible outside Colab
demo.launch(share=True, debug=False)

In [ ]:
import gradio as gr
import time
import torch

# Ensure left-padding is active for clean generation
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

def chat_tpu(message, history, max_new_tokens, temperature):
    conversation = []

    # Adapt to Gradio's history format (dict or tuple list)
    for turn in history:
        if isinstance(turn, dict):
            conversation.append({"role": turn.get("role", "user"), "content": turn.get("content", "")})
        elif isinstance(turn, (list, tuple)) and len(turn) == 2:
            user_msg, bot_msg = turn
            if user_msg:
                conversation.append({"role": "user", "content": user_msg})
            if bot_msg:
                conversation.append({"role": "assistant", "content": bot_msg})

    conversation.append({"role": "user", "content": message})

    formatted_prompt = tokenizer.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

    start_time = time.time()
    with torch.no_grad():
        output_tokens = model.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            do_sample=(temperature > 0.0),
            temperature=float(temperature) if temperature > 0.0 else None,
            pad_token_id=tokenizer.eos_token_id
        )
    elapsed = time.time() - start_time

    new_tokens = output_tokens[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    speed = len(new_tokens) / elapsed if elapsed > 0 else 0
    print(f"TPU Inference: {len(new_tokens)} tokens generated in {elapsed:.2f}s ({speed:.1f} tok/s)")

    return response

# Clean ChatInterface without invalid theme parameters
demo = gr.ChatInterface(
    fn=chat_tpu,
    title="Cloud TPU v5e-1 Inference Dashboard",
    description="Interactive chatbot running on Google Cloud TPU v5e hardware via native PyTorch-XLA.",
    additional_inputs=[
        gr.Slider(minimum=20, maximum=256, value=100, step=10, label="Max New Tokens"),
        gr.Slider(minimum=0.0, maximum=1.0, value=0.0, step=0.1, label="Temperature (0 = Greedy)")
    ]
)

# Launch with a public shareable tunnel URL
demo.launch(share=True, debug=False)

In [ ]:
# 1. Install dependencies
!pip install -q fastapi uvicorn pyngrok transformers accelerate torch nest-asyncio

import os
import torch
import nest_asyncio
import uvicorn
from fastapi import FastAPI, Request
from pyngrok import ngrok
from transformers import AutoModelForCausalLM, AutoTokenizer

# Allow uvicorn to run inside Colab's existing asyncio event loop
nest_asyncio.apply()

# 2. Ngrok Authentication
# Get your free token at: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTHTOKEN = "3EM7xuROaPavm7b1bYjWQRPnCBq_43uDM6FKujvNEBAfpEyDN"
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# 3. Load Qwen2.5 (1.5B fits entirely in T4 GPU VRAM and loads in ~30s)
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading model: {MODEL_ID} onto GPU...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Model loaded successfully on:", model.device)

# 4. OpenAI-Compatible API
app = FastAPI(title="Remote LLM Engine")

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID, "device": str(model.device)}

@app.post("/v1/chat/completions")
async def chat_completions(req: Request):
    body = await req.json()
    messages = body.get("messages", [])

    # Format using Qwen chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=body.get("max_tokens", 400),
            temperature=body.get("temperature", 0.2),
            do_sample=False
        )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, outputs)
    ]
    reply = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return {
        "id": "chatcmpl-colab-qwen",
        "object": "chat.completion",
        "model": body.get("model", MODEL_ID),
        "choices": [
            {
                "index": 0,
                "message": {"role": "assistant", "content": reply.strip()},
                "finish_reason": "stop"
            }
        ]
    }

# 5. Open Tunnel and Serve
# Close any existing ngrok tunnels if restarting cell
ngrok.kill()

public_url = ngrok.connect(8000).public_url
print("\n" + "=" * 60)
print(f"🚀 YOUR LIVE TPU/GPU ENDPOINT: {public_url}/v1")
print("=" * 60 + "\n")

uvicorn.run(app, host="0.0.0.0", port=8000)

In [ ]:
# 1. Install dependencies
!pip install -q fastapi uvicorn pyngrok transformers accelerate torch nest-asyncio

import os
import torch
import nest_asyncio
import uvicorn
from fastapi import FastAPI, Request
from pyngrok import ngrok
from transformers import AutoModelForCausalLM, AutoTokenizer

nest_asyncio.apply()

# 2. Ngrok Authentication
NGROK_AUTHTOKEN = "3EM7xuROaPavm7b1bYjWQRPnCBq_43uDM6FKujvNEBAfpEyDN"
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# 3. Load Qwen2.5 (1.5B fits cleanly on T4 GPU)
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading {MODEL_ID} onto GPU...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Model loaded successfully on:", model.device)

# 4. OpenAI-Compatible FastAPI Server
app = FastAPI(title="Remote LLM Engine")

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID, "device": str(model.device)}

@app.post("/v1/chat/completions")
async def chat_completions(req: Request):
    body = await req.json()
    messages = body.get("messages", [])

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=body.get("max_tokens", 350),
            temperature=body.get("temperature", 0.2),
            do_sample=False
        )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, outputs)
    ]
    reply = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return {
        "id": "chatcmpl-colab-qwen",
        "object": "chat.completion",
        "model": body.get("model", MODEL_ID),
        "choices": [
            {
                "index": 0,
                "message": {"role": "assistant", "content": reply.strip()},
                "finish_reason": "stop"
            }
        ]
    }

# 5. Connect Tunnel and Start Serving
ngrok.kill()
public_url = ngrok.connect(8000).public_url

print("\n" + "=" * 60)
print(f"🚀 YOUR LIVE TPU/GPU ENDPOINT: {public_url}/v1")
print("=" * 60 + "\n")

uvicorn.run(app, host="0.0.0.0", port=8000)

In [ ]:
# 1. Install dependencies
!pip install -q fastapi uvicorn pyngrok transformers accelerate torch nest-asyncio

import os
import torch
import nest_asyncio
import uvicorn
from fastapi import FastAPI, Request
from pyngrok import ngrok, conf
from transformers import AutoModelForCausalLM, AutoTokenizer

nest_asyncio.apply()

# 2. Ngrok Authentication (Bypass if already configured)
NGROK_AUTHTOKEN = "3EM7xuROaPavm7b1bYjWQRPnCBq_43uDM6FKujvNEBAfpEyDN"
try:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
except Exception:
    pass

# Ensure pyngrok uses the token explicitly
conf.get_default().auth_token = NGROK_AUTHTOKEN

# 3. Load Qwen2.5 (1.5B fits cleanly on T4 GPU)
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading {MODEL_ID} onto GPU...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Model loaded successfully on:", model.device)

# 4. OpenAI-Compatible FastAPI Server
app = FastAPI(title="Remote LLM Engine")

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID, "device": str(model.device)}

@app.post("/v1/chat/completions")
async def chat_completions(req: Request):
    body = await req.json()
    messages = body.get("messages", [])

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=body.get("max_tokens", 350),
            temperature=body.get("temperature", 0.2),
            do_sample=False
        )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, outputs)
    ]
    reply = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return {
        "id": "chatcmpl-colab-qwen",
        "object": "chat.completion",
        "model": body.get("model", MODEL_ID),
        "choices": [
            {
                "index": 0,
                "message": {"role": "assistant", "content": reply.strip()},
                "finish_reason": "stop"
            }
        ]
    }

# 5. Connect Tunnel and Start Serving
ngrok.kill()
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print("\n" + "=" * 60)
print(f"🚀 YOUR LIVE TPU/GPU ENDPOINT: {public_url}/v1")
print("=" * 60 + "\n")

uvicorn.run(app, host="0.0.0.0", port=8000)

In [ ]:
# 1. Install dependencies
!pip install -q fastapi uvicorn pyngrok transformers accelerate torch

import os
import threading
import torch
import uvicorn
from fastapi import FastAPI, Request
from pyngrok import ngrok, conf
from transformers import AutoModelForCausalLM, AutoTokenizer

# 2. Ngrok Authentication
NGROK_AUTHTOKEN = "3EM7xuROaPavm7b1bYjWQRPnCBq_43uDM6FKujvNEBAfpEyDN"
try:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
except Exception:
    pass
conf.get_default().auth_token = NGROK_AUTHTOKEN

# 3. Load Qwen2.5 (1.5B fits cleanly on T4 GPU)
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading {MODEL_ID} onto GPU...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Model loaded successfully on:", model.device)

# 4. OpenAI-Compatible FastAPI Server
app = FastAPI(title="Remote LLM Engine")

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID, "device": str(model.device)}

@app.post("/v1/chat/completions")
async def chat_completions(req: Request):
    body = await req.json()
    messages = body.get("messages", [])

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=body.get("max_tokens", 350),
            temperature=body.get("temperature", 0.2),
            do_sample=False
        )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, outputs)
    ]
    reply = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return {
        "id": "chatcmpl-colab-qwen",
        "object": "chat.completion",
        "model": body.get("model", MODEL_ID),
        "choices": [
            {
                "index": 0,
                "message": {"role": "assistant", "content": reply.strip()},
                "finish_reason": "stop"
            }
        ]
    }

# 5. Connect Tunnel and Start Uvicorn in a Thread
ngrok.kill()
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print("\n" + "=" * 60)
print(f"🚀 YOUR LIVE TPU/GPU ENDPOINT: {public_url}/v1")
print("=" * 60 + "\n")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)

server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
print("✅ Uvicorn server is running in the background!")

In [2]:
# 1. Dependencies
!pip install -q fastapi uvicorn pyngrok transformers accelerate torch

import asyncio
import os
import threading
import torch
import uvicorn
from fastapi import FastAPI, Request
from pyngrok import ngrok, conf
from transformers import AutoModelForCausalLM, AutoTokenizer

# 2. Ngrok Authtoken
NGROK_AUTHTOKEN = "3EM7xuROaPavm7b1bYjWQRPnCBq_43uDM6FKujvNEBAfpEyDN"
try:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
except Exception:
    pass
conf.get_default().auth_token = NGROK_AUTHTOKEN

# 3. Model Setup
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading {MODEL_ID} onto GPU...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Model loaded successfully on:", model.device)

# 4. App setup
app = FastAPI(title="Remote LLM Engine")

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID, "device": str(model.device)}

@app.post("/v1/chat/completions")
async def chat_completions(req: Request):
    body = await req.json()
    messages = body.get("messages", [])

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=body.get("max_tokens", 350),
            temperature=body.get("temperature", 0.2),
            do_sample=False
        )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, outputs)
    ]
    reply = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return {
        "id": "chatcmpl-colab-qwen",
        "object": "chat.completion",
        "model": body.get("model", MODEL_ID),
        "choices": [
            {
                "index": 0,
                "message": {"role": "assistant", "content": reply.strip()},
                "finish_reason": "stop"
            }
        ]
    }

# 5. Connect Tunnel & Start Event Loop in Thread
ngrok.kill()
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print("\n" + "=" * 60)
print(f"🚀 YOUR LIVE TPU/GPU ENDPOINT: {public_url}/v1")
print("=" * 60 + "\n")

def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("✅ Uvicorn server is running and listening on port 8000!")

Loading Qwen/Qwen2.5-1.5B-Instruct onto GPU...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully on: cuda:0

🚀 YOUR LIVE TPU/GPU ENDPOINT: https://atop-remarry-richly.ngrok-free.dev/v1

✅ Uvicorn server is running and listening on port 8000!
